<a href="https://colab.research.google.com/github/charmy-patel/practicals/blob/bigdata/PIG_HIVE_multiplePartition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Now this workflow fully mimics:

Pig → ETL (cleaning, transformations).

Hive → SQL analytics with multi-level partitions.

PySpark → one unified pipeline that does both.

<!-- Now workflow fully mimics:

Pig → ETL (cleaning, transformations).

Hive → SQL analytics with multi-level partitions.

PySpark → one unified pipeline that does both. -->

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, sum as spark_sum, avg, year, month

# 1️⃣ Start Spark with Hive support
spark = SparkSession.builder \
    .appName("PigHiveExample") \
    .enableHiveSupport() \
    .getOrCreate()

# 2️⃣ Load raw data (CSV for example)
raw_df = spark.read.csv("/content/drive/My Drive/Colab Notebooks/clickstream.csv", header=True, inferSchema=True)

# Sample columns: user_id, timestamp, page, device_type, purchase_amount

# 3️⃣ Data Cleaning & Transformation (Pig-like)

clean_df = raw_df \
    .filter(col("purchase_amount").isNotNull()) \
    .withColumn("timestamp", to_timestamp(col("timestamp"), "dd-MM-yyyy HH:mm")) \
    .filter(col("timestamp").isNotNull()) \
    .filter(col("device_type").isin("mobile", "desktop")) \
    .withColumn("year", year(col("timestamp"))) \
    .withColumn("month", month(col("timestamp")))

# 4️⃣ Save cleaned data into Hive with multiple partitions
clean_df.write \
    .mode("overwrite") \
    .partitionBy("year", "month", "device_type") \
    .format("parquet") \
    .saveAsTable("ecommerce_cleaned_partitioned")



# 5️⃣ Run Hive-style query (partition pruning will make it faster!)
result_df = spark.sql("""
    SELECT year, month, device_type,
           COUNT(DISTINCT user_id) AS unique_users,
           SUM(purchase_amount) AS total_sales,
           AVG(purchase_amount) AS avg_purchase
    FROM ecommerce_cleaned_partitioned
    WHERE year = 2025 AND month = 1  -- partition pruning
    GROUP BY year, month, device_type
    ORDER BY total_sales DESC
""")

# 6️⃣ Show results
result_df.show()

+----+-----+-----------+------------+-----------------+-----------------+
|year|month|device_type|unique_users|      total_sales|     avg_purchase|
+----+-----+-----------+------------+-----------------+-----------------+
|2025|    1|     mobile|           8|           897.69|        112.21125|
|2025|    1|    desktop|           8|689.8199999999999|86.22749999999999|
+----+-----+-----------+------------+-----------------+-----------------+

